# TrustLine — Tier 2 SaaS on Google Colab

Colab acts as the **regional SaaS** (ap-south): Postgres (RLS), NATS JetStream, nginx mTLS ingest, control plane,
ingest-api, workers (health + AIOps) and the dashboard API. Your **laptop stays Tier 1**: simulator + collector.

```
laptop: simulator -> collector --mTLS 1.3--> bore.pub:PORT (raw TCP relay) --> Colab nginx:8443 -> ingest -> NATS -> workers -> Postgres
browser --> https://xxxx.trycloudflare.com --> Colab api:8000 (dashboard + REST + WebSocket)
```

The bore relay only forwards TCP, so TLS ends **inside Colab** — the relay sees ciphertext only.

**Before you start (on the laptop):**
1. `python scripts/tl.py init --local` (once)
2. Regenerate certs so the server cert also names `bore.pub`:
   `EXTRA_SAN=DNS:bore.pub FORCE=1 sh deploy/certs/gen.sh` (Git Bash)
3. `python scripts/native.py bundle` → upload `dist/trustline-colab.zip` below.

**Limits:** sessions end after ~12 h or ~90 min idle; data is lost when the runtime resets; keep this tab open.

In [ ]:
#@title 1. Upload the bundle (dist/trustline-colab.zip from your laptop)
import os, zipfile
from google.colab import files
os.chdir('/content')
up = files.upload()
name = next(iter(up))
with zipfile.ZipFile(name) as z:
    z.extractall('/content/trustline')
os.chdir('/content/trustline')
print('extracted', len(os.listdir('.')), 'entries; .env present:', os.path.exists('.env'),
      '| dashboard built:', os.path.exists('dashboard/dist/index.html'))

In [ ]:
#@title 2. Install (Postgres, nginx, NATS, tunnels, Python deps) - about 2 min
%cd /content/trustline
import sys
sys.path.insert(0, 'scripts')
import native
native.setup_linux(timescale=False)   # True tries TimescaleDB; plain Postgres works too
print('setup done')

In [ ]:
#@title 3. Start the SaaS + tunnels (runs in the background)
import subprocess, time, re, pathlib
%cd /content/trustline
log = open('/content/saas.log', 'w')
proc = subprocess.Popen(['python', 'scripts/native.py', 'saas', '--tunnel'], stdout=log, stderr=subprocess.STDOUT)
for _ in range(120):
    txt = pathlib.Path('/content/saas.log').read_text(errors='replace')
    if 'On your laptop run' in txt or 'failed' in txt or proc.poll() is not None:
        break
    time.sleep(1)
print(txt[-2500:])

In [ ]:
#@title 4. Status / logs
!tail -n 5 /content/trustline/.run/logs/{ingest,workers,api}.log
!curl -s localhost:8000/api/healthz; echo
!su postgres -c "psql -d trustline -tAc \"select count(*) as payloads from received_payloads\""

In [ ]:
#@title 5. Keep-alive (optional): prints a heartbeat every minute so the runtime looks busy
import time, urllib.request, json
for i in range(600):
    try:
        h = json.load(urllib.request.urlopen('http://127.0.0.1:8000/api/healthz', timeout=3))
        print(time.strftime('%H:%M:%S'), h)
    except Exception as e:
        print('api not answering:', e)
    time.sleep(60)

In [ ]:
#@title 6. Stop
!python scripts/native.py stop; nginx -c /content/trustline/.run/nginx.conf -p /content/trustline/.run -s stop